In [18]:
import getpass
import os

try:
    from dotenv import load_dotenv
    # 加载环境变量
    load_dotenv()
except ImportError:
    # 如果 python-dotenv 未安装，则定义一个空函数作为回退
    def load_dotenv():
        print("load_dotenv not found")
        pass

def check_smith():
    os.environ["LANGSMITH_TRACING"] = "true"
    if "LANGSMITH_API_KEY" not in os.environ:
        os.environ["LANGSMITH_API_KEY"] = getpass.getpass(
            prompt="Enter your LangSmith API key (optional): "
        )
    if "LANGSMITH_PROJECT" not in os.environ:
        os.environ["LANGSMITH_PROJECT"] = getpass.getpass(
            prompt='Enter your LangSmith Project Name (default = "default"): '
        )
        if not os.environ.get("LANGSMITH_PROJECT"):
            os.environ["LANGSMITH_PROJECT"] = "default"
    print("check smith OK")


def check_openai_key():
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")
    print("check openai key OK")

def check_tavily_key():
    if not os.environ.get("TAVILY_API_KEY"):
        os.environ["TAVILY_API_KEY"] = getpass.getpass("Enter API key for TAVILY: ")
    print("check tavily key OK")

check_smith()
check_openai_key()
check_tavily_key()

check smith OK
check openai key OK
check tavily key OK


In [19]:
# Import relevant functionality
from langchain_tavily import TavilySearch

search = TavilySearch(max_results=2)
search_results = search.invoke("What is the weather in Shanghai")
print(search_results)

{'query': 'What is the weather in Shanghai', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'title': 'Weather in Shanghai', 'url': 'https://www.weatherapi.com/', 'content': "{'location': {'name': 'Shanghai', 'region': 'Shanghai', 'country': 'China', 'lat': 31.005, 'lon': 121.4086, 'tz_id': 'Asia/Shanghai', 'localtime_epoch': 1753166703, 'localtime': '2025-07-22 14:45'}, 'current': {'last_updated_epoch': 1753165800, 'last_updated': '2025-07-22 14:30', 'temp_c': 33.2, 'temp_f': 91.8, 'is_day': 1, 'condition': {'text': 'Partly cloudy', 'icon': '//cdn.weatherapi.com/weather/64x64/day/116.png', 'code': 1003}, 'wind_mph': 14.5, 'wind_kph': 23.4, 'wind_degree': 128, 'wind_dir': 'SE', 'pressure_mb': 1009.0, 'pressure_in': 29.8, 'precip_mm': 0.0, 'precip_in': 0.0, 'humidity': 63, 'cloud': 75, 'feelslike_c': 41.1, 'feelslike_f': 106.0, 'windchill_c': 31.6, 'windchill_f': 89.0, 'heatindex_c': 37.2, 'heatindex_f': 99.0, 'dewpoint_c': 23.8, 'dewpoint_f': 74.8, 'vis_km': 10.

In [24]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

model = init_chat_model("gpt-4.1-nano", model_provider="openai")
query = "Hi!"
response = model.invoke([{"role": "user", "content": query}])
print(response.text())

Hello! How can I assist you today?


In [54]:

search = TavilySearch(max_results=2)
tools = [search]
# create_react_agent 内部调用.bind_tools方法
agent_executor = create_react_agent(model, tools)

input_message = {"role": "user", "content": "Hi!"}
response = agent_executor.invoke({"messages": [input_message]})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hi!
================================== Ai Message ==================================

Hello! How can I assist you today?


In [63]:
# Create the agent
memory = MemorySaver()
agent_executor = create_react_agent(model, tools, checkpointer=memory)
# Use the agent
config = {"configurable": {"thread_id": "abc123"}}

input_message = {
    "role": "user",
    "content": "Hi, 我是Bob, 我生活在上海.",
}
# 流输出
for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Hi, 我是Bob, 我生活在上海.
================================== Ai Message ==================================

你好，Bob！很高兴认识你。你在上海生活一定很精彩。如果你有任何问题或者需要帮助的地方，随时告诉我！


In [65]:
input_message = {
    "role": "user",
    "content": "我生活的地方天气怎么样?",
}
for step in agent_executor.stream(
    {"messages": [input_message]}, config, stream_mode="values"
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

我生活的地方天气怎么样?
================================== Ai Message ==================================
Tool Calls:
  tavily_search (call_c89dlzKdharZ38IRAz6Snt97)
 Call ID: call_c89dlzKdharZ38IRAz6Snt97
  Args:
    query: 上海当前天气
================================= Tool Message =================================
Name: tavily_search

{"query": "上海当前天气", "follow_up_questions": null, "answer": null, "images": [], "results": [{"url": "http://sh.cma.gov.cn/sh/tqyb/qxsk/", "title": "天气实况-上海市气象局", "content": "当前暂无预警！ · 07月26日星期六 · 阴. 阵雨 · 28℃～32℃ · 东北风 风力6-7级.", "score": 0.72231215, "raw_content": null}, {"url": "https://weather.cma.cn/web/weather/58367.html", "title": "中国气象局-天气预报-城市预报- 上海", "content": "| é£å | ä¸åé£ | ä¸åé£ | ä¸åé£ | ä¸åé£ | ä¸åé£ | ä¸åé£ | ä¸åé£ | ä¸åé£ | | éæ°´ | 7.2mm | 8.8mm | 1.2mm | 0.2mm | æ éæ°´ | 2.3mm | 2.3mm | 2.3mm | | é£å | ä¸åé£ | ä¸åé£ | è¥¿å

In [66]:

for step, metadata in agent_executor.stream(
    {"messages": [input_message]}, config=config, stream_mode="messages"
):
    if metadata["langgraph_node"] == "agent" and (text := step.text()):
        print(text, end="|")

你|生活|的|上海|目前|天气|是|阴|天|，有|阵|雨|，|气|温|在|28|℃|到|32|℃|之间|，|东北|风|风|力|6|-|7|级|。|请|根据|天气|情况|做好|相|应|的|准备|。如果|你|需要|更|详细|的|天气|信息|，可以|告诉|我|！|

In [74]:
agent_executor = create_react_agent(model, [TavilySearch(max_results=2)], checkpointer=memory)
input_message = {
    "role": "user",
    "content": "上海的天气",
}
config = {"configurable": {"thread_id": "xyz123"}}
# python3.11也可使用.astream_events流回token令牌
async for event in agent_executor.astream_events(
    {"messages": [input_message]},config=config
):
    kind = event["event"]
    if kind == "on_chain_start":
        if event["name"] == "Agent":
        # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print(f"Starting agent: {event['name']} with input: {event['data'].get('input')}")
    elif kind == "on_chain_end":
        if event["name"] == "Agent":
            # Was assigned when creating the agent with `.with_config({"run_name": "Agent"})`
            print()
            print("--")
            print(f"Done agent: {event['name']} with output: {event['data'].get('output')['output']}")
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")
    elif kind == "on_tool_start":
        print("--")
        print(f"Starting tool: {event['name']} with inputs: {event['data'].get('input')}")
    elif kind == "on_tool_end":
        print(f"Done tool: {event['name']}")
        print(f"Tool output was: {event['data'].get('output')}")
        print("--")

--
Starting tool: tavily_search with inputs: {'query': '上海的天气'}
Done tool: tavily_search
Tool output was: content="{'error': ClientConnectorCertificateError(ConnectionKey(host='api.tavily.com', port=443, is_ssl=True, ssl=True, proxy=None, proxy_auth=None, proxy_headers_hash=None), SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1020)'))}" name='tavily_search' tool_call_id='call_GkTtS5Gk3Od2S6wYQHG3w4bE'
--
抱|歉|，由|于|技术|原因|，我|无法|获取|到|最新|的|上海|天气|信息|。|你|可以|尝|试|使用|天气|预|报|网站|或|应用|获取|最新|的|天气|情况|。|需要|我|帮|你|查|找|相关|的|天气|预|报|网站|吗|？|

In [27]:
# 绑定工具
model_with_tools = model.bind_tools(tools)
query = "Hi!"
response = model_with_tools.invoke([{"role": "user", "content": query}])
# 此处工具不会调用
print(f"Message content: {response.text()}")
print(f"Tool calls: {response.tool_calls}\n")

query = "Search for the weather in SF"
response = model_with_tools.invoke([{"role": "user", "content": query}])
# 此处将调用工具
print(f"Message content: {response.text()}\n")
print(f"Tool calls: {response.tool_calls}")

Message content: Hello! How can I assist you today?
Tool calls: []

Message content: 

Tool calls: [{'name': 'tavily_search', 'args': {'query': 'weather in San Francisco'}, 'id': 'call_k4SUZEkvlvwm643FLLa1Gq14', 'type': 'tool_call'}]
